In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets torchvision
except Exception:
    pass

In [2]:
import warnings

import numpy as np
import torch
from maite_datasets.object_detection import SeaDrone
from torchvision.models import ResNet18_Weights, resnet18

from dataeval import Embeddings, Metadata
from dataeval.config import set_device, set_seed, use_load_threads, use_max_processes
from dataeval.core import image_stats
from dataeval.data import Limit, Shuffle, View
from dataeval.exceptions import ExperimentalWarning
from dataeval.extractors import TorchExtractor

set_seed(0)  # For reproducibility
set_device("cuda" if torch.cuda.is_available() else "cpu")

# This guide hides the warnings that load threads and `record_reads` are experimental, and the ones about binning
# factors it does not use.
warnings.filterwarnings("ignore", category=ExperimentalWarning)
warnings.filterwarnings("ignore", message=".*(was dropped|binned automatically).*")

In [3]:
# A 48-image sample of SeaDrone stands in for a dataset that decodes each photo as it is read: by default its loader
# decodes the full 3840 x 2160 image on every read.
dataset = View(SeaDrone(root="./data", image_set="val", download=True), [Shuffle(seed=0), Limit(48)])

In [4]:
weights = ResNet18_Weights.DEFAULT
extractor = TorchExtractor(
    resnet18(weights=weights, progress=False),
    transforms=weights.transforms(),
    layer_name="avgpool",
)
view = View(dataset, record_reads=True)

with use_load_threads(4):
    embeddings = Embeddings(view, extractor=extractor, batch_size=32).compute()

print(embeddings.shape)

(48, 512)


In [5]:
# TEST ASSERTION CELL ###
import threading


class _Counting:
    """Pass-through that records each read and the thread it ran on."""

    def __init__(self, source):
        self.source, self.metadata, self.reads, self.threads = source, source.metadata, 0, set()

    def __len__(self):
        return len(self.source)

    def __getitem__(self, index):
        self.reads += 1
        self.threads.add(threading.get_ident())
        return self.source[index]


serial = np.asarray(Embeddings(dataset, extractor=extractor, batch_size=32).compute())
assert np.allclose(np.asarray(embeddings), serial, atol=1e-5)

counted = _Counting(dataset)
with use_load_threads(4):
    Embeddings(View(counted), extractor=extractor, batch_size=32).compute()
assert len(counted.threads) > 1

In [6]:
metadata = Metadata(view)
print(metadata.level_counts)

{'unit': 48, 'instance': 305}


In [7]:
# TEST ASSERTION CELL ###
assert metadata.dataframe.equals(Metadata(dataset).dataframe)

recorded = _Counting(dataset)
recorded_view = View(recorded, record_reads=True)
with use_load_threads(4):
    Embeddings(recorded_view, extractor=extractor, batch_size=32).compute()
before = recorded.reads
_ = Metadata(recorded_view).dataframe
assert recorded.reads - before <= 1

plain = _Counting(dataset)
plain_view = View(plain)
Embeddings(plain_view, extractor=extractor, batch_size=32).compute()
before = plain.reads
_ = Metadata(plain_view).dataframe
assert plain.reads - before >= len(dataset)

In [8]:
with use_max_processes(4):
    stats = image_stats(view)

print(len(stats["source_index"]), "rows of", len(stats["stats"]), "statistics")

353 rows of 39 statistics


In [9]:
# TEST ASSERTION CELL ###
sample = View(dataset, Limit(4))
default = image_stats(sample)
with use_max_processes(4):
    parallel = image_stats(sample)
for name, values in default["stats"].items():
    a, b = np.asarray(values), np.asarray(parallel["stats"][name])
    assert np.array_equal(a, b, equal_nan=a.dtype.kind in "fc"), name